# Stage 4 G1 — Correlation pre-test for Qwen3.6-35B-A3B + L23 SAE on SuperGPQA

**Goal**: verify that the contrastive SAE features at L23 predict SuperGPQA correctness on held-out data *before* committing GPU hours to RL.

**Mirror of the Qwen3.5-4B G1 that produced ρ=0.540.** Two phases:

1. **Discovery** (n=50 SuperGPQA responses): find top-10 helpful + top-10 harmful features via `(mean|correct − mean|wrong) / pooled_std`
2. **Validation** (n=100 held-out, disjoint): compute `R_mech = sum(helpful acts) − sum(harmful acts)`, mean-pooled over response tokens, then Spearman ρ vs outcome.

**Pass threshold**: ρ ≥ 0.30 → go to S4-G2 RL. ρ ≥ 0.50 is excellent (Qwen3.5-4B baseline). ρ < 0.10 → layer/SAE insufficient; revisit.

**Inputs**:
- Qwen3.6-35B-A3B base model (~70 GB bf16)
- SAE checkpoint at `/content/drive/MyDrive/mechreward/s4_qwen36/sae_l23/step_21972.pt` (or latest step_*.pt in that dir)
- SuperGPQA dataset, easy+middle difficulty, seed=42

**Disjoint from S4-L0**: uses questions `[100:250]` from the same shuffled+filtered set. S4-L0 used `[0:100]`. Zero overlap.

**Budget**: ~30-60 min on RTX 6000 Blackwell 96 GB (inference only).

## Cell 1 — Install (same stack as S4-L0 / G3)

In [ ]:
import sys, subprocess, os, shutil, site
from pathlib import Path

def _pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check)

def _check():
    try:
        import transformers
        try:
            from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
        except ImportError:
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
        return 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, transformers.__version__
    except Exception as e:
        return False, f'import-error: {e}'

ok, ver = _check()
print(f'Current: transformers={ver}, qwen3_5_moe={ok}')

TRANSFORMERS_REF = 'main'
SRC_DIR = '/content/transformers_src'

if not ok:
    print(f'\n→ Installing transformers @ {TRANSFORMERS_REF}')
    _pip('install', '-q',
         'accelerate', 'datasets',
         'huggingface_hub==1.5.0',
         'safetensors', 'einops', 'scikit-learn',
         'sentencepiece', 'tokenizers', 'protobuf', 'scipy')
    _pip('uninstall', '-y', 'transformers', check=False)
    _pip('uninstall', '-y', 'transformers', check=False)
    for p in sys.path + site.getsitepackages() + [site.getusersitepackages()]:
        tr = Path(p) / 'transformers'
        if tr.exists():
            shutil.rmtree(tr, ignore_errors=True)
    if os.path.exists(SRC_DIR):
        shutil.rmtree(SRC_DIR)
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/huggingface/transformers.git', SRC_DIR], check=True)
    if TRANSFORMERS_REF != 'main':
        subprocess.run(['git', '-C', SRC_DIR, 'checkout', '--quiet', TRANSFORMERS_REF], check=True)
    _pip('install', '--force-reinstall', '--no-deps', '--no-cache-dir', SRC_DIR)
    _pip('install', '--no-cache-dir', 'flash-linear-attention', 'causal-conv1d', check=False)
    for mod in list(sys.modules):
        if mod.startswith('transformers') or mod.startswith('huggingface_hub'):
            del sys.modules[mod]
    ok, ver = _check()
    print(f'Post-install: transformers={ver}, qwen3_5_moe={ok}')
    if not ok:
        print('\n⚠️  Restart kernel and re-run this cell.')
        raise SystemExit

# HF auth + Drive
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('✅ HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

from google.colab import drive
drive.mount('/content/drive')

import json, time, random, re, gc
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import spearmanr, pearsonr

DRIVE_ROOT = Path('/content/drive/MyDrive/mechreward/s4_qwen36')
G1_ROOT = DRIVE_ROOT / 'g1'
G1_ROOT.mkdir(parents=True, exist_ok=True)
print(f'\nDrive root: {DRIVE_ROOT}')
print(f'G1 artifacts: {G1_ROOT}')

## Cell 2 — CFG

Uses the SAE checkpoint at step_21972 (90M tokens, var_exp ~0.835). Point to a later checkpoint if training went longer.

In [ ]:
# Locate SAE checkpoint — latest by default, or override with explicit path
SAE_CKPT_DIR = DRIVE_ROOT / 'sae_l23'
SAE_CKPT_OVERRIDE = None  # e.g. SAE_CKPT_DIR / 'step_21972.pt' to force a specific one

if SAE_CKPT_OVERRIDE and SAE_CKPT_OVERRIDE.exists():
    sae_ckpt_path = SAE_CKPT_OVERRIDE
else:
    ckpts = sorted(SAE_CKPT_DIR.glob('step_*.pt'),
                   key=lambda p: int(re.match(r'step_(\d+)', p.stem).group(1)))
    assert ckpts, f'No step_*.pt in {SAE_CKPT_DIR}'
    sae_ckpt_path = ckpts[-1]
print(f'SAE checkpoint: {sae_ckpt_path}  ({sae_ckpt_path.stat().st_size/1e6:.0f} MB)')

CFG = dict(
    model_id='Qwen/Qwen3.6-35B-A3B',
    sae_ckpt=str(sae_ckpt_path),
    sae_layer=23,
    # Dataset
    difficulty_filter=['easy', 'middle'],
    s4l0_n=100,           # questions S4-L0 used (skip these)
    discovery_n=50,       # for feature discovery
    validation_n=100,     # for ρ measurement
    max_new_tokens=3000,
    seed=42,
    # Pack selection
    n_helpful=10,
    n_harmful=10,
    # Decision gate
    rho_pass=0.30,
    rho_excellent=0.50,
)
print(json.dumps({k: v for k, v in CFG.items() if k != 'sae_ckpt'}, indent=2))

## Cell 3 — Load Qwen3.6-35B-A3B (skip if already in memory)

In [ ]:
# If you're coming from the training notebook in the SAME session, `model` may still be loaded.
# In that case, free training-specific buffers first. Otherwise load fresh.

import transformers
try:
    from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
except ImportError:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
assert 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, 'Restart kernel, rerun Cell 1.'

from transformers import AutoTokenizer, AutoModelForImageTextToText

existing_model = 'model' in dir() and hasattr(globals().get('model'), 'named_parameters')
if existing_model:
    print('Found existing `model` in kernel. Reusing it.')
    print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
else:
    tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForImageTextToText.from_pretrained(
        CFG['model_id'],
        dtype=torch.bfloat16,
        device_map='cuda',
        attn_implementation='sdpa',
        trust_remote_code=True,
    )
    model.eval()
    print(f'Model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Make sure tok exists
if 'tok' not in dir():
    tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

# Free training tensors if they exist (from SAE training notebook in same session)
for name in ['sae', 'buffer', 'optimizer', 'scheduler', 'opt', 'sched', 'acts', 'x']:
    if name in dir():
        try:
            del globals()[name]
        except Exception:
            pass
gc.collect(); torch.cuda.empty_cache()
print(f'\nVRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 4 — Load SAE from Drive checkpoint

Our training script (`scripts/train_sae_qwen35.py`) saves `step_N.pt` with keys including `W_enc`, `W_dec`, `b_enc`, `b_dec`, `k`, plus optimizer/scheduler state. We only need the SAE tensors.

In [ ]:
sae_state = torch.load(CFG['sae_ckpt'], map_location='cuda', weights_only=False)
if hasattr(sae_state, 'state_dict'):
    sae_state = sae_state.state_dict()

def _get(d, *keys, default=None):
    for k in keys:
        if k in d:
            return d[k]
        # Accept nested 'sae' or 'model' prefix
        for prefix in ('sae', 'model'):
            kk = f'{prefix}.{k}'
            if kk in d:
                return d[kk]
    return default

W_enc = _get(sae_state, 'W_enc').to('cuda', torch.float32)
W_dec = _get(sae_state, 'W_dec').to('cuda', torch.float32)
# Ensure shape [d_model, d_sae] for W_enc
if W_enc.shape[0] < W_enc.shape[1]:
    pass  # already [d_model, d_sae]
else:
    W_enc = W_enc.t().contiguous()
# Ensure shape [d_sae, d_model] for W_dec
if W_dec.shape[0] > W_dec.shape[1]:
    pass  # already [d_sae, d_model]
else:
    W_dec = W_dec.t().contiguous()

d_model, d_sae = W_enc.shape
b_enc = _get(sae_state, 'b_enc', default=torch.zeros(d_sae, device='cuda')).to('cuda', torch.float32)
b_dec = _get(sae_state, 'b_dec', default=torch.zeros(d_model, device='cuda')).to('cuda', torch.float32)
K = int(_get(sae_state, 'k', 'K', default=128))

print(f'SAE: d_model={d_model}, d_sae={d_sae}, k={K}')
print(f'  W_enc {tuple(W_enc.shape)}  W_dec {tuple(W_dec.shape)}')
print(f'  b_enc norm={b_enc.norm().item():.3f}  b_dec norm={b_dec.norm().item():.3f}')

del sae_state
gc.collect(); torch.cuda.empty_cache()

## Cell 5 — Layer locator + TopK SAE encode

Same pattern as S4-L0. TopK is critical: matches exactly what the SAE was trained on. Without it we'd be using a ReLU SAE.

In [ ]:
def get_layer_module(m, idx):
    candidates = [m]
    if hasattr(m, 'base_model') and m.base_model is not m:
        candidates.append(m.base_model.model if hasattr(m.base_model, 'model') else m.base_model)
    if hasattr(m, 'model'):
        candidates.append(m.model)
    paths = [
        ('model', 'language_model', 'layers'),
        ('language_model', 'layers'),
        ('model', 'layers'),
        ('layers',),
    ]
    for start in candidates:
        for path in paths:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except (IndexError, TypeError): continue
    raise RuntimeError(f'Could not locate layer {idx}')

_layer = get_layer_module(model, CFG['sae_layer'])
print(f'L{CFG["sae_layer"]}: {type(_layer).__name__}')

class HiddenCapture:
    def __init__(self): self.h = None
    def hook(self, module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        self.h = h.detach()

def register_capture(m, layer_idx):
    cap = HiddenCapture()
    handle = get_layer_module(m, layer_idx).register_forward_hook(cap.hook)
    return cap, handle

@torch.inference_mode()
def topk_encode(h: torch.Tensor) -> torch.Tensor:
    """h: [T, d_model] or [B, T, d_model] bf16/fp32. Returns [..., d_sae] in fp32 with exactly K non-zero per row (after ReLU)."""
    h = h.to(torch.float32)
    h = h - b_dec
    # Chunk over tokens to keep memory low for long sequences
    orig_shape = h.shape
    flat = h.reshape(-1, d_model)
    T = flat.shape[0]
    CHUNK = 512
    outs = []
    for i in range(0, T, CHUNK):
        pre = flat[i:i+CHUNK] @ W_enc + b_enc  # [chunk, d_sae]
        topv, topi = torch.topk(pre, K, dim=-1)  # [chunk, K]
        out = torch.zeros_like(pre)
        out.scatter_(-1, topi, topv)
        out = F.relu(out)
        outs.append(out)
    return torch.cat(outs, dim=0).reshape(*orig_shape[:-1], d_sae)

# Sanity: random vector encode gives L0 ~= K
with torch.inference_mode():
    probe_h = torch.randn(8, d_model, device='cuda')
    probe_acts = topk_encode(probe_h)
    l0 = (probe_acts > 0).float().sum(dim=-1)
print(f'L0 on random input: {l0.tolist()}  (expected ~{K})')

## Cell 6 — SuperGPQA dataset (disjoint from S4-L0)

S4-L0 used `[0:100]` of the shuffled+filtered set (seed=42, easy+middle). We use `[100:100+150]` — first 50 for discovery, next 100 for validation. Zero overlap with S4-L0.

In [ ]:
from datasets import load_dataset

raw = load_dataset('m-a-p/SuperGPQA', split='train')
filtered = raw.filter(lambda ex: ex['difficulty'] in CFG['difficulty_filter'])
shuffled = filtered.shuffle(seed=CFG['seed'])
total_needed = CFG['s4l0_n'] + CFG['discovery_n'] + CFG['validation_n']
assert len(shuffled) >= total_needed, f'Not enough questions: have {len(shuffled)}, need {total_needed}'

discovery = shuffled.select(range(CFG['s4l0_n'], CFG['s4l0_n'] + CFG['discovery_n']))
validation = shuffled.select(range(CFG['s4l0_n'] + CFG['discovery_n'],
                                    CFG['s4l0_n'] + CFG['discovery_n'] + CFG['validation_n']))
print(f'Discovery set:  {len(discovery)} questions (indices [{CFG["s4l0_n"]}:{CFG["s4l0_n"]+CFG["discovery_n"]}])')
print(f'Validation set: {len(validation)} questions (indices [{CFG["s4l0_n"]+CFG["discovery_n"]}:{CFG["s4l0_n"]+CFG["discovery_n"]+CFG["validation_n"]}])')
print(f'\nSample discovery question:')
ex = discovery[0]
print(f'  discipline: {ex["discipline"]}/{ex["field"]}  difficulty: {ex["difficulty"]}')
print(f'  Q: {ex["question"][:180]}')
print(f'  gold letter: {ex["answer_letter"]}  (#options: {len(ex["options"])})')

## Cell 7 — Prompt template + MCQ verifier (same as S4-L0)

In [ ]:
def format_prompt(ex):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(ex['options']))
    msgs = [{
        'role': 'user',
        'content': (
            f"Answer the following multiple-choice question. Reason step by step, "
            f"then put the letter of the correct answer in \\boxed{{}}.\n\n"
            f"Question: {ex['question']}\n\n"
            f"Options:\n{choices}"
        )
    }]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
LETTER_AT_END_RE = re.compile(r'[Aa]nswer[:\s]+([A-J])\b')

def extract_letter(completion, n_opts):
    m = list(BOXED_RE.finditer(completion))
    if m:
        letter = m[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts: return letter
    m2 = list(LETTER_AT_END_RE.finditer(completion))
    if m2:
        letter = m2[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts: return letter
    tail = completion[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands:
        letter = cands[-1]
        if ord(letter) - ord('A') < n_opts: return letter
    return None

def verify(ex, completion):
    pred = extract_letter(completion, len(ex['options']))
    return pred is not None and pred == ex['answer_letter']

## Cell 8 — Discovery phase — generate + mean-pool SAE acts per question

For each of the 50 discovery questions: greedy generate → full-sequence forward with hook at L23 → TopK encode → mean-pool over response tokens → store `[d_sae]`. Result: matrix `[50, d_sae]` + outcome labels.

In [ ]:
from tqdm.auto import tqdm

model.config.use_cache = True

def run_and_encode(dataset, desc):
    """Generate + encode + mean-pool per question. Returns (acts [N, d_sae], outcomes, metadata)."""
    cap, handle = register_capture(model, CFG['sae_layer'])
    pooled_acts = []
    outcomes = []
    meta = []
    t0 = time.time()
    try:
        for ex in tqdm(dataset, desc=desc):
            prompt = format_prompt(ex)
            enc = tok(prompt, return_tensors='pt').to('cuda')
            P = enc.input_ids.shape[1]
            with torch.no_grad():
                gen = model.generate(
                    **enc,
                    max_new_tokens=CFG['max_new_tokens'],
                    do_sample=False,
                    pad_token_id=tok.pad_token_id,
                    use_cache=True,
                )
            # Second forward on full sequence to capture hidden at L23
            cap.h = None
            attn_mask = (gen != tok.pad_token_id).long()
            with torch.no_grad():
                model(input_ids=gen, attention_mask=attn_mask, use_cache=False)
            hidden = cap.h[0]  # [T, d_model]
            response_mask = attn_mask[0].clone()
            response_mask[:P] = 0
            n_resp = int(response_mask.sum().item())
            # Encode only response tokens
            if n_resp == 0:
                pooled = torch.zeros(d_sae, device='cuda')
            else:
                response_h = hidden[response_mask.bool()]  # [n_resp, d_model]
                acts = topk_encode(response_h)              # [n_resp, d_sae]
                pooled = acts.mean(dim=0)                   # [d_sae]
            pooled_acts.append(pooled.cpu())
            completion = tok.decode(gen[0][P:], skip_special_tokens=True)
            is_correct = verify(ex, completion)
            outcomes.append(int(is_correct))
            meta.append({
                'discipline': ex['discipline'],
                'difficulty': ex['difficulty'],
                'gold': ex['answer_letter'],
                'pred': extract_letter(completion, len(ex['options'])),
                'n_gen_tokens': int(gen.shape[1] - P),
                'n_resp_tokens_used': n_resp,
                'correct': is_correct,
            })
            del hidden, gen, attn_mask
    finally:
        handle.remove()
    acts_tensor = torch.stack(pooled_acts)  # [N, d_sae]
    print(f'{desc}: {time.time()-t0:.0f}s, accuracy {sum(outcomes)}/{len(outcomes)} = {100*sum(outcomes)/len(outcomes):.0f}%')
    return acts_tensor, np.array(outcomes), meta

disc_acts, disc_outcomes, disc_meta = run_and_encode(discovery, 'Discovery (50Q)')
torch.save({'acts': disc_acts, 'outcomes': disc_outcomes, 'meta': disc_meta},
           G1_ROOT / 'discovery_captures.pt')
print(f'✅ Saved discovery captures: {G1_ROOT}/discovery_captures.pt')
print(f'Correct/wrong split: {disc_outcomes.sum()}/{(1-disc_outcomes).sum()}')

## Cell 9 — Contrastive feature discovery + build pack

For each feature: `effect_size = (mean(acts|correct) − mean(acts|wrong)) / pooled_std`. Top-10 positive = helpful; top-10 most-negative = harmful.

In [ ]:
X = disc_acts.numpy()  # [N, d_sae]
y = disc_outcomes.astype(bool)

if y.sum() < 5 or (~y).sum() < 5:
    print(f'⚠️  Very unbalanced split ({y.sum()} correct / {(~y).sum()} wrong). Discovery will be noisy.')
    print('Options: increase discovery_n, or widen difficulty filter.')

h_c = X[y]      # [n_c, d_sae]
h_w = X[~y]
mean_c = h_c.mean(axis=0)
mean_w = h_w.mean(axis=0)
var_c = h_c.var(axis=0, ddof=1) if y.sum() > 1 else np.zeros(d_sae)
var_w = h_w.var(axis=0, ddof=1) if (~y).sum() > 1 else np.zeros(d_sae)
pooled_std = np.sqrt((var_c + var_w) / 2) + 1e-8
effect_size = (mean_c - mean_w) / pooled_std  # [d_sae]

# Filter out features that never fire on either side (effect undefined)
ever_active = ((h_c > 0).any(axis=0) | (h_w > 0).any(axis=0))
effect_size_filtered = np.where(ever_active, effect_size, 0.0)

# Top-10 most positive (helpful) and most negative (harmful)
sorted_idx = np.argsort(effect_size_filtered)  # ascending
harmful_ids = sorted_idx[:CFG['n_harmful']].tolist()
helpful_ids = sorted_idx[::-1][:CFG['n_helpful']].tolist()

print(f'{"=" * 60}')
print(f'HELPFUL PACK (top-{CFG["n_helpful"]} positive effect size):')
print(f'{"=" * 60}')
for fid in helpful_ids:
    print(f'  F{fid:>6}  d={effect_size_filtered[fid]:+.3f}  mean_c={mean_c[fid]:.4f}  mean_w={mean_w[fid]:.4f}')
print(f'\n{"=" * 60}')
print(f'HARMFUL PACK (top-{CFG["n_harmful"]} most-negative effect size):')
print(f'{"=" * 60}')
for fid in harmful_ids:
    print(f'  F{fid:>6}  d={effect_size_filtered[fid]:+.3f}  mean_c={mean_c[fid]:.4f}  mean_w={mean_w[fid]:.4f}')

ALL_FEATS = torch.tensor(helpful_ids + harmful_ids, dtype=torch.long, device='cuda')
FEAT_WEIGHTS = torch.tensor([1.0]*len(helpful_ids) + [-1.0]*len(harmful_ids), dtype=torch.float32, device='cuda')
print(f'\nPack built: {len(helpful_ids)} helpful + {len(harmful_ids)} harmful')

## Cell 10 — Validation phase — 100 disjoint questions

Generate + encode on validation set, compute `R_mech = sum(helpful) − sum(harmful)` mean-pooled over response tokens per question. Identical encode pipeline as discovery (TopK, full response tokens).

In [ ]:
val_acts, val_outcomes, val_meta = run_and_encode(validation, 'Validation (100Q)')
torch.save({'acts': val_acts, 'outcomes': val_outcomes, 'meta': val_meta},
           G1_ROOT / 'validation_captures.pt')
print(f'✅ Saved validation captures: {G1_ROOT}/validation_captures.pt')

## Cell 11 — Compute R_mech per question + Spearman/Pearson correlation

In [ ]:
# R_mech per validation question using the discovered pack
val_X = val_acts.numpy()  # [100, d_sae]
helpful_arr = np.array(helpful_ids)
harmful_arr = np.array(harmful_ids)
R_helpful = val_X[:, helpful_arr].sum(axis=1)
R_harmful = val_X[:, harmful_arr].sum(axis=1)
R_mech = R_helpful - R_harmful  # [100]

rho, rho_p = spearmanr(R_mech, val_outcomes)
r, r_p = pearsonr(R_mech, val_outcomes)

print(f'{"=" * 60}')
print(f'STAGE GATE 1 — VALIDATION RESULTS')
print(f'{"=" * 60}')
print(f'Validation set: {len(val_outcomes)} questions')
print(f'Baseline accuracy: {val_outcomes.mean()*100:.1f}% ({val_outcomes.sum()}/{len(val_outcomes)})')
print(f'')
print(f'Spearman ρ = {rho:+.4f}  (p={rho_p:.2e})')
print(f'Pearson  r = {r:+.4f}  (p={r_p:.2e})')
print(f'')
print(f'Pass threshold: ρ ≥ {CFG["rho_pass"]:.2f}')
print(f'Excellent (Qwen3.5-4B level): ρ ≥ {CFG["rho_excellent"]:.2f}  (theirs: 0.540)')
print(f'')
if rho >= CFG['rho_excellent']:
    verdict = 'EXCELLENT — matches or beats Qwen3.5-4B. Proceed to S4-G2 RL.'
    emoji = '🏆'
elif rho >= CFG['rho_pass']:
    verdict = f'PASS — features predict correctness. Proceed to S4-G2 RL.'
    emoji = '✅'
elif rho >= 0.10:
    verdict = 'WEAK — signal exists but below threshold. Options: more SAE training (100-200M more tokens), try different layer, or widen discovery to 100+ questions.'
    emoji = '⚠️'
else:
    verdict = 'FAIL — no correlation. Layer choice may be wrong, or features not captured. Consider retraining SAE at different layer (peak was L22, currently at L23).'
    emoji = '❌'
print(f'{emoji}  {verdict}')

## Cell 12 — Save reasoning_pack + G1 summary

In [ ]:
# Pull mean_correct / mean_wrong for each selected feature (from discovery stats)
def _feat_entry(fid, weight):
    return {
        'feature_id': int(fid),
        'effect_size': float(effect_size_filtered[fid]),
        'mean_correct': float(mean_c[fid]),
        'mean_wrong': float(mean_w[fid]),
        'weight': float(weight),
        'validated': True,
    }

pack = {
    'name': 'qwen3.6-35b-a3b/reasoning_pack',
    'version': '0.1.0',
    'model_name': CFG['model_id'],
    'release': f'caiovicentino1/Qwen3.6-35B-A3B-SAE-L{CFG["sae_layer"]}-topk',
    'sae_id': f'layer_{CFG["sae_layer"]}',
    'description': (
        f"Contrastive reasoning features for Qwen3.6-35B-A3B at residual stream layer {CFG['sae_layer']}. "
        f"Discovered via contrastive correctness analysis on {CFG['discovery_n']} held-out SuperGPQA responses "
        f"(thinking mode, (easy+middle) difficulty filter). "
        f"Validated by Stage Gate 1 correlation test: Spearman ρ={rho:.3f} (p={rho_p:.2e}) "
        f"on {CFG['validation_n']} held-out SuperGPQA questions (disjoint from discovery and from S4-L0 probe set). "
        f"These features predict whether the model will answer a science/reasoning question correctly, "
        f"and are intended for use as a dense reward signal in RL fine-tuning (mechreward thesis)."
    ),
    'helpful_features': [_feat_entry(fid, +1.0) for fid in helpful_ids],
    'harmful_features': [_feat_entry(fid, -1.0) for fid in harmful_ids],
    'validation': {
        'method': 'Stage Gate 1 — passive correlation pre-test',
        'date': time.strftime('%Y-%m-%d'),
        'benchmark': 'SuperGPQA',
        'difficulty_filter': CFG['difficulty_filter'],
        'discovery_n': int(CFG['discovery_n']),
        'discovery_split': f'[{CFG["s4l0_n"]}:{CFG["s4l0_n"]+CFG["discovery_n"]}]',
        'n_questions': int(CFG['validation_n']),
        'validation_split': f'[{CFG["s4l0_n"]+CFG["discovery_n"]}:{CFG["s4l0_n"]+CFG["discovery_n"]+CFG["validation_n"]}]',
        'n_correct': int(val_outcomes.sum()),
        'spearman_rho': float(rho),
        'spearman_p': float(rho_p),
        'pearson_r': float(r),
        'pearson_p': float(r_p),
        'reward_formula': f'R_mech = sum(helpful_activations) - sum(harmful_activations) per response, mean-pooled over tokens at L{CFG["sae_layer"]}',
        'pass_threshold': CFG['rho_pass'],
        'verdict': (
            'PASS-EXCELLENT' if rho >= CFG['rho_excellent']
            else 'PASS' if rho >= CFG['rho_pass']
            else 'WEAK' if rho >= 0.10
            else 'FAIL'
        ),
        'sae_checkpoint': str(Path(CFG['sae_ckpt']).name),
    },
    'metadata': {
        'architecture': 'qwen3_5_moe_triple_hybrid',  # Qwen3.6 is branded as 3.6 but internally qwen3_5_moe
        'd_model': int(d_model),
        'd_sae': int(d_sae),
        'k': int(K),
        'num_hidden_layers': 40,
        'target_layer': int(CFG['sae_layer']),
        'peak_contrastive_layer': 22,  # from S4-L0
        'training_tokens': None,  # will be filled manually based on ckpt step
        'hf_sae_repo': f'caiovicentino1/Qwen3.6-35B-A3B-SAE-L{CFG["sae_layer"]}-topk',
    },
}

# Save to Drive
pack_path = G1_ROOT / 'reasoning_pack.json'
with open(pack_path, 'w') as f:
    json.dump(pack, f, indent=2)
print(f'✅ Saved pack: {pack_path}')

# Save G1 summary (leaner, for figures/ and memory)
summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model': CFG['model_id'],
    'sae_ckpt': str(Path(CFG['sae_ckpt']).name),
    'sae_layer': CFG['sae_layer'],
    'discovery': {
        'n': CFG['discovery_n'],
        'n_correct': int(disc_outcomes.sum()),
        'n_wrong': int((1-disc_outcomes).sum()),
        'baseline_accuracy': float(disc_outcomes.mean()),
    },
    'validation': {
        'n': CFG['validation_n'],
        'n_correct': int(val_outcomes.sum()),
        'baseline_accuracy': float(val_outcomes.mean()),
        'spearman_rho': float(rho),
        'spearman_p': float(rho_p),
        'pearson_r': float(r),
        'pearson_p': float(r_p),
    },
    'decision': {
        'verdict': pack['validation']['verdict'],
        'pass_threshold': CFG['rho_pass'],
        'next_action': (
            'Proceed to S4-G2 (tiny RL sanity)' if rho >= CFG['rho_pass']
            else 'Continue SAE training or re-evaluate layer choice'
        ),
    },
    'helpful_ids': helpful_ids,
    'harmful_ids': harmful_ids,
}
summary_path = G1_ROOT / 's4_g1_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✅ Saved summary: {summary_path}')
print()
print(json.dumps(summary['decision'], indent=2))

## Cell 13 (optional) — Copy pack to mechreward repo catalog

If ρ passes, copy the pack to `catalogs/qwen3.6-35b-a3b/reasoning_pack.json` so the S4-G2 notebook and the `mechreward` library can load it by name. Manual git-add + commit after.

In [ ]:
if rho >= CFG['rho_pass']:
    if not Path('/content/mechreward').exists():
        subprocess.run(['git', 'clone', '-q', 'https://github.com/caiovicentino/mechreward.git', '/content/mechreward'], check=True)
    dest = Path('/content/mechreward/catalogs/qwen3.6-35b-a3b')
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(pack_path, dest / 'reasoning_pack.json')
    print(f'✅ Copied pack → {dest}/reasoning_pack.json')
    print(f'\nNext: commit + push to github.com/caiovicentino/mechreward, then run S4-G2.')
else:
    print(f'Skipping catalog copy (ρ={rho:.3f} < pass threshold {CFG["rho_pass"]}).')
    print('Rerun after more SAE training, or adjust methodology.')